# Step 3 · Supervised fine-tuning (SFT)

Train a LoRA adapter on question → approved answer pairs with TRL's SFTTrainer.

*Same as the dashboard page, done in code.*

In [ ]:
# ── Setup (run first) ────────────────────────────────────────────────────────────
# Uses the same lab registry as the dashboard. To experiment in a sandbox instead:
#   os.environ["LAB_HOME"] = "/tmp/my_lab"   (before importing lab)
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # models are already cached locally
import pandas as pd
from lab import data, registry, paths, rewards
data.bootstrap_samples()
BASE = registry.ensure_base(paths.DEFAULT_BASE_MODEL)
if not registry.rewards():
    registry.add_reward(rewards.DEFAULT_RULES, note="Default draft rules")

def latest(kind):
    """Newest model id of a kind: base | sft | rl | rlhf | rm (or None)."""
    ms = [m for m in registry.models() if m["kind"] == kind]
    return ms[-1]["id"] if ms else None

def latest_ds(role):
    d = registry.latest_dataset(role)
    return d["id"] if d else None

print("Lab home:", paths.home(), "| base model:", BASE)

In [ ]:
from lab import jobs
cfg = dict(jobs.SFT_PRESETS["Tiny (smoke test)"])
cfg

In [ ]:
# Runs `python -m lab.train <run>` in the background and waits for it (about 15 s on a GTX 1080 Ti)
rid = jobs.start("sft", BASE, latest_ds("sft_train"), latest_ds("sft_val"), cfg, wait=True)
run = registry.run(rid)
print(run["status"], "→ new model", run["output_model_id"], run["error"] or "")

In [ ]:
m = pd.DataFrame(jobs.read_jsonl(run["metrics_path"]))
m[[c for c in ("step", "loss", "eval_loss") if c in m]].dropna(how="all", subset=[c for c in ("loss", "eval_loss") if c in m])

What the trainer actually does (the lab's own code):

In [ ]:
import inspect
from lab import train
print(inspect.getsource(train.run_sft))

In [ ]:
from lab import engines
SFT = run["output_model_id"]
engines.ask(SFT, "Can I promise a customer their ladder arrives tomorrow?")